# Лабораторная работа: классификация ЭКГ. Часть 2 — классификатор

В этой части вы обучите нейросетевой классификатор диагнозов на датасете, который собрали и экспортировали в первой части (`ecg-warehouse/notebooks/lab_dataset.ipynb`), оцените его на тестовой выборке и попробуете улучшить базовую модель.

Обучение построено на PyTorch Lightning и библиотеке `ecg_ml`: `WarehouseDatasetModule` загружает экспорт `ecg-warehouse`, `ClassifierLModule` реализует шаги обучения, валидации и теста.

Задача **многометочная** (multi-label): классы не взаимоисключающие, у одной записи может быть несколько кодов варианта сразу (например, `IMI` и `ASMI`) или ни одного. Поэтому у сети по одному независимому выходу на класс, а не softmax по классам.

Ноутбук выполняется в окружении проекта `ecg-ml`: установите его командой `uv sync` в корне репозитория и выберите ядро из `.venv`.

In [13]:
import os
from pathlib import Path

# Пути в ноутбуке отсчитываются от корня репозитория, а Jupyter может запустить ноутбук из notebooks/.
if Path.cwd().name == "notebooks":
    os.chdir("..")

### Параметры

- `VARIANT` — номер вашего варианта, тот же, что в первой части;
- `DATASET_PATH` — каталог, в который первая часть экспортировала датасет (путь относительно корня репозитория `ecg-ml`);
- `LEARNING_RATE`, `BATCH_SIZE`, `EPOCHS` — шаг обучения, размер батча и максимальное число эпох;
- `NUM_WORKERS` — число процессов, которые готовят батчи.

In [14]:
VARIANT = 2
DATASET_PATH = f"data/ptb-xl-variant-{VARIANT}"

LEARNING_RATE = 0.1
BATCH_SIZE = 32
NUM_WORKERS = 2
EPOCHS = 30

In [15]:
from functools import partial
import torch
from lightning import Trainer
from ecg_ml.data.warehouse import WarehouseDatasetModule
from ecg_ml.modules.classifier import ClassifierLModule

## 1. Данные

`WarehouseDatasetModule` читает `signals.npz` и `meta.csv` из экспорта. Тестовая выборка — часть `test` из разбиения хранилища. Валидационная выделяется из `train` тоже по пациентам, около 10 % пациентов; `random_state` фиксирует это разбиение, чтобы модели сравнивались на одних и тех же записях.

Классы и их порядок берутся из экспорта: `CLASSES` совпадёт со списком из первой части.

In [16]:
dataset = WarehouseDatasetModule(DATASET_PATH, random_state=0)
datamodule = dataset.to_datamodule(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)

CLASSES = dataset.classes
CLASSES

['NORM', 'CLBBB', 'CRBBB']

## 2. Модель

В ячейке ниже — заготовка `ECGClassifier`.

**Задание.** Замените реализацию `ECGClassifier` в ячейке ниже собственной архитектурой нейросети для классификации ЭКГ. Сохраните имя класса и параметр `output_size`: модель создаётся в ячейке с `ClassifierLModule` как `ECGClassifier(output_size=len(CLASSES))`. Требования к модели:

- вход — батч сигналов формы `(batch, 12, 5000)`: 12 отведений по 5000 отсчётов;
- выход — логиты формы `(batch, len(CLASSES))`.

In [17]:
class ResBlock(torch.nn.Module):
    def __init__(self, c_in, c_out, stride=1, k=7, p_drop=0.2):
        super().__init__()
        self.main = torch.nn.Sequential(
            torch.nn.Conv1d(c_in, c_out, k, stride, padding=k // 2, bias=False),
            torch.nn.BatchNorm1d(c_out),
            torch.nn.ReLU(),
            torch.nn.Dropout(p_drop),
            torch.nn.Conv1d(c_out, c_out, k, padding=k // 2, bias=False),
            torch.nn.BatchNorm1d(c_out),
        )
        if stride != 1 or c_in != c_out:
            self.shortcut = torch.nn.Sequential(
                torch.nn.Conv1d(c_in, c_out, 1, stride, bias=False),
                torch.nn.BatchNorm1d(c_out),
            )
        else:
            self.shortcut = torch.nn.Identity()

    def forward(self, x):
        return torch.relu(self.main(x) + self.shortcut(x))


class ECGClassifier(torch.nn.Module):
    def __init__(self, output_size=2):
        super().__init__()
        self.stem = torch.nn.Sequential(
            torch.nn.Conv1d(12, 32, 15, stride=2, padding=7, bias=False),
            torch.nn.BatchNorm1d(32),
            torch.nn.ReLU(),
        )
        self.blocks = torch.nn.Sequential(
            ResBlock(32, 32, stride=1),
            ResBlock(32, 64, stride=2),
            ResBlock(64, 96, stride=2),
            ResBlock(96, 128, stride=2),
            ResBlock(128, 128, stride=2),
        )
        self.head = torch.nn.Sequential(
            torch.nn.Dropout(0.3),
            torch.nn.Linear(2 * 128, output_size),
        )

    def forward(self, x):
        x = self.blocks(self.stem(x))
        x = torch.cat([x.mean(dim=-1), x.amax(dim=-1)], dim=1)
        return self.head(x)

### Функция потерь и оптимизатор

- `BCEWithLogitsLoss` — бинарная кросс-энтропия, посчитанная независимо для каждого класса; подходит для многометочной задачи;
- `Adam` с шагом обучения `LEARNING_RATE`;
- `ReduceLROnPlateau` уменьшает шаг в 10 раз, если `val/loss` не улучшается 10 эпох подряд.

In [18]:
optimizer = partial(torch.optim.Adam, lr=LEARNING_RATE)
lr_scheduler = {
    "scheduler": partial(
        torch.optim.lr_scheduler.ReduceLROnPlateau, mode="min", factor=0.1, patience=10
    ),
    "monitor": "val/loss",
    "interval": "epoch",
    "frequency": 1,
}
criterion = torch.nn.BCEWithLogitsLoss()

`ClassifierLModule` объединяет сеть, функцию потерь, оптимизатор и метрики. На валидации и тесте считаются precision, recall и F1 для каждого класса и их macro-среднее.

In [19]:
classifier = ClassifierLModule(
    optimizer=optimizer,
    lr_scheduler=lr_scheduler,
    criterion=criterion,
    network=ECGClassifier(output_size=len(CLASSES)),
    labels=CLASSES,
)

c:\Users\User\Desktop\ML_Labs\ecg-ml\.venv\Lib\site-packages\lightning\pytorch\utilities\parsing.py:213: Attribute 'criterion' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['criterion'])`.
c:\Users\User\Desktop\ML_Labs\ecg-ml\.venv\Lib\site-packages\lightning\pytorch\utilities\parsing.py:213: Attribute 'network' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['network'])`.


## 3. Обучение

`Trainer` проводит валидацию после каждой эпохи и обучает не больше `EPOCHS` эпох в смешанной точности (`precision=16`). Метрики по эпохам сохраняются в `logs/lab/lightning_logs/version_N/metrics.csv`, веса — в `logs/lab/lightning_logs/version_N/checkpoints/`. Каталог `logs/` исключён из git.

In [20]:
trainer = Trainer(
    accelerator="gpu",
    devices=1,
    precision="16-mixed",
    check_val_every_n_epoch=1,
    min_epochs=1,
    max_epochs=EPOCHS,
    default_root_dir="logs/lab",
)

Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


In [21]:
trainer.fit(
    model=classifier,
    datamodule=datamodule,
)

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
c:\Users\User\Desktop\ML_Labs\ecg-ml\.venv\Lib\site-packages\lightning\pytorch\utilities\model_summary\model_summary.py:242: Precision 16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.


┏━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name                  ┃ Type                      ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ network               │ ECGClassifier             │  641 K │ train │     0 │
│ 1 │ criterion             │ BCEWithLogitsLoss         │      0 │ train │     0 │
│ 2 │ val_metrics           │ MetricCollection          │      0 │ train │     0 │
│ 3 │ val_loss              │ MeanMetric                │      0 │ train │     0 │
│ 4 │ val_loss_best         │ MinMetric                 │      0 │ train │     0 │
│ 5 │ val_f1_macro_best     │ MaxMetric                 │      0 │ train │     0 │
│ 6 │ test_metrics          │ MetricCollection          │      0 │ train │     0 │
│ 7 │ test_confusion_matrix │ MultilabelConfusionMatrix │      0 │ train │     0 │
└───┴───────────────────────┴───────────────────────────┴────────┴───────┴───────┘

Trainable params: 641 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 641 K                                                                                                
Total estimated model params size (MB): 2.564                                                                      
Modules in train mode: 81                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

c:\Users\User\Desktop\ML_Labs\ecg-ml\.venv\Lib\site-packages\lightning\pytorch\utilities\_pytree.py:21: 
`isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` 
instead.

c:\Users\User\Desktop\ML_Labs\ecg-ml\.venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py
:429: Consider setting `persistent_workers=True` in 'val_dataloader' to speed up the dataloader worker 
initialization.

c:\Users\User\Desktop\ML_Labs\ecg-ml\.venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py
:429: Consider setting `persistent_workers=True` in 'train_dataloader' to speed up the dataloader worker 
initialization.

c:\Users\User\Desktop\ML_Labs\ecg-ml\.venv\Lib\site-packages\torchmetrics\utilities\prints.py:43: UserWarning: 
Encountered `nan` values in tensor. Will be removed.
  warnings.warn(*args, **kwargs)

`Trainer.fit` stopped: `max_epochs=30` reached.


## 4. Оценка на тестовой выборке

`trainer.test` считает метрики на test для весов после последней эпохи и печатает `classification_report` из scikit-learn: precision, recall и F1 по каждому классу. Тестовую выборку используйте только для итоговой оценки, а модели и гиперпараметры сравнивайте по валидации.

In [22]:
trainer.test(
    model=classifier,
    datamodule=datamodule,
)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
c:\Users\User\Desktop\ML_Labs\ecg-ml\.venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:429: Consider setting `persistent_workers=True` in 'test_dataloader' to speed up the dataloader worker initialization.


Output()

precision    recall  f1-score   support

        NORM       0.00      0.00      0.00      1911
       CLBBB       0.00      0.00      0.00       130
       CRBBB       0.00      0.00      0.00       110

   micro avg       0.00      0.00      0.00      2151
   macro avg       0.00      0.00      0.00      2151
weighted avg       0.00      0.00      0.00      2151
 samples avg       0.00      0.00      0.00      2151

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│       test/f1/CLBBB       │            0.0            │
│       test/f1/CRBBB       │            0.0            │
│       test/f1/NORM        │            0.0            │
│       test/f1/macro       │            0.0            │
│   test/precision/CLBBB    │            0.0            │
│   test/precision/CRBBB    │            0.0            │
│    test/precision/NORM    │            0.0            │
│   test/precision/macro    │            0.0            │
│     test/recall/CLBBB     │            0.0            │
│     test/recall/CRBBB     │            0.0            │
│     test/recall/NORM      │            0.0            │
│     test/recall/macro     │            0.0            │
└───────────────────────────┴───────────────────────────┘

[{'test/f1/NORM': 0.0,
  'test/f1/CLBBB': 0.0,
  'test/f1/CRBBB': 0.0,
  'test/f1/macro': 0.0,
  'test/precision/NORM': 0.0,
  'test/precision/CLBBB': 0.0,
  'test/precision/CRBBB': 0.0,
  'test/precision/macro': 0.0,
  'test/recall/NORM': 0.0,
  'test/recall/CLBBB': 0.0,
  'test/recall/CRBBB': 0.0,
  'test/recall/macro': 0.0}]

## 5. Улучшение модели

Что можно попробовать для улучшения:

- подобрать `LEARNING_RATE`, `BATCH_SIZE`, `EPOCHS`;
- учесть дисбаланс классов: `torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor(dataset.pos_weight))`;
- изменить архитектуру `ECGClassifier`: число и ширину слоёв, размер ядер свёрток, нормализацию, dropout.

In [23]:
LEARNING_RATE = 0.001

criterion = torch.nn.BCEWithLogitsLoss(
    pos_weight=torch.as_tensor(
        dataset.pos_weight,
        dtype=torch.float32
    )
)

optimizer = partial(torch.optim.Adam, lr=LEARNING_RATE)

classifier = ClassifierLModule(
    optimizer=optimizer,
    lr_scheduler=lr_scheduler,
    criterion=criterion,
    network=ECGClassifier(output_size=len(CLASSES)),
    labels=CLASSES,
)

trainer = Trainer(
    accelerator="gpu",
    devices=1,
    precision="16-mixed",
    check_val_every_n_epoch=1,
    min_epochs=1,
    max_epochs=EPOCHS,
    default_root_dir="logs/lab",
)

trainer.fit(
    model=classifier,
    datamodule=datamodule,
)

c:\Users\User\Desktop\ML_Labs\ecg-ml\.venv\Lib\site-packages\lightning\pytorch\utilities\parsing.py:213: Attribute 'criterion' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['criterion'])`.
c:\Users\User\Desktop\ML_Labs\ecg-ml\.venv\Lib\site-packages\lightning\pytorch\utilities\parsing.py:213: Attribute 'network' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['network'])`.
Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litm

┏━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name                  ┃ Type                      ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ network               │ ECGClassifier             │  641 K │ train │     0 │
│ 1 │ criterion             │ BCEWithLogitsLoss         │      0 │ train │     0 │
│ 2 │ val_metrics           │ MetricCollection          │      0 │ train │     0 │
│ 3 │ val_loss              │ MeanMetric                │      0 │ train │     0 │
│ 4 │ val_loss_best         │ MinMetric                 │      0 │ train │     0 │
│ 5 │ val_f1_macro_best     │ MaxMetric                 │      0 │ train │     0 │
│ 6 │ test_metrics          │ MetricCollection          │      0 │ train │     0 │
│ 7 │ test_confusion_matrix │ MultilabelConfusionMatrix │      0 │ train │     0 │
└───┴───────────────────────┴───────────────────────────┴────────┴───────┴───────┘

Trainable params: 641 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 641 K                                                                                                
Total estimated model params size (MB): 2.564                                                                      
Modules in train mode: 81                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

c:\Users\User\Desktop\ML_Labs\ecg-ml\.venv\Lib\site-packages\lightning\pytorch\utilities\_pytree.py:21: 
`isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` 
instead.

c:\Users\User\Desktop\ML_Labs\ecg-ml\.venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py
:429: Consider setting `persistent_workers=True` in 'val_dataloader' to speed up the dataloader worker 
initialization.

c:\Users\User\Desktop\ML_Labs\ecg-ml\.venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py
:429: Consider setting `persistent_workers=True` in 'train_dataloader' to speed up the dataloader worker 
initialization.

`Trainer.fit` stopped: `max_epochs=30` reached.


In [24]:
trainer.test(
    model=classifier,
    datamodule=datamodule,
)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

c:\Users\User\Desktop\ML_Labs\ecg-ml\.venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:429: Consider setting `persistent_workers=True` in 'test_dataloader' to speed up the dataloader worker initialization.


precision    recall  f1-score   support

        NORM       0.82      0.94      0.87      1911
       CLBBB       0.80      0.98      0.88       130
       CRBBB       0.71      1.00      0.83       110

   micro avg       0.81      0.94      0.87      2151
   macro avg       0.78      0.97      0.86      2151
weighted avg       0.81      0.94      0.87      2151
 samples avg       0.46      0.46      0.46      2151

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│       test/f1/CLBBB       │    0.8819444179534912     │
│       test/f1/CRBBB       │    0.8333333134651184     │
│       test/f1/NORM        │    0.8733219504356384     │
│       test/f1/macro       │    0.8628665208816528     │
│   test/precision/CLBBB    │    0.8037974834442139     │
│   test/precision/CRBBB    │    0.7142857313156128     │
│    test/precision/NORM    │    0.8183897733688354     │
│   test/precision/macro    │    0.7788243293762207     │
│     test/recall/CLBBB     │    0.9769230484962463     │
│     test/recall/CRBBB     │            1.0            │
│     test/recall/NORM      │     0.936159074306488     │
│     test/recall/macro     │    0.9710273742675781     │
└───────────────────────────┴───────────────────────────┘

[{'test/f1/NORM': 0.8733219504356384,
  'test/f1/CLBBB': 0.8819444179534912,
  'test/f1/CRBBB': 0.8333333134651184,
  'test/f1/macro': 0.8628665208816528,
  'test/precision/NORM': 0.8183897733688354,
  'test/precision/CLBBB': 0.8037974834442139,
  'test/precision/CRBBB': 0.7142857313156128,
  'test/precision/macro': 0.7788243293762207,
  'test/recall/NORM': 0.936159074306488,
  'test/recall/CLBBB': 0.9769230484962463,
  'test/recall/CRBBB': 1.0,
  'test/recall/macro': 0.9710273742675781}]

## Выводы

### Модель

В качестве модели использована небольшая одномерная ResNet. Входной сигнал `(batch, 12, 5000)` проходит через свёртку-«стем» (ядро 15, stride 2) и пять остаточных блоков с двумя свёртками (ядро 7) и BatchNorm. Четыре блока из пяти понижают временное разрешение (stride 2) и увеличивают число каналов с 32 до 128, поэтому сеть видит как форму отдельных комплексов, так и ритм на всём 10-секундном отрезке. В голове используются одновременно среднее и максимум по времени (`mean` и `amax`), затем Dropout и линейный слой, дающий по одному логиту на каждый из трёх классов. Для многометочной постановки применяется `BCEWithLogitsLoss`.

### Почему при `LEARNING_RATE = 0.1` модель не работает

При шаге 0.1  обучение расходится. Диагностика (была проведена отдельно, не включена в итоговый файл) показала, что все параметры сети, включая первый свёрточный слой, превратились в `NaN`, при этом входные сигналы и метки в проверенном батче были корректны. Значит, проблема не в исходных данных. Модель с `NaN`-весами выдаёт `NaN`-логиты, поэтому не предсказывает ни одного положительного класса, и precision, recall и F1 на тесте равны нулю для всех классов, включая самый частый `NORM`. Вероятная причина - слишком крупный шаг. Результат воспроизвёлся и на более простой модели, и на ResNet. Стоит отметить, что простого изменения LEARNING_RATE без прочих улучшений уже хватило, чтобы модель начала показывать приемлемые результаты.

### Результат после улучшений

Улучшения: `LEARNING_RATE = 0.001` и взвешивание положительного класса через `pos_weight` из `dataset.pos_weight`. Итоги на тестовой выборке:

| Класс | Precision | Recall | F1 |
|---|---|---|---|
| NORM | 0.82 | 0.94 | 0.87 |
| CLBBB | 0.80 | 0.98 | 0.88 |
| CRBBB | 0.71 | 1.00 | 0.83 |
| macro | 0.78 | 0.97 | 0.86 |

Macro F1 вырос с 0.00 до 0.86, модель уверенно находит все три состояния, в том числе редкие `CLBBB` и `CRBBB`. Recall заметно выше precision (0.97 против 0.78): это ожидаемое следствие `pos_weight`, который штрафует пропуск положительного класса сильнее, чем ложное срабатывание, поэтому модель склонна «перестраховываться». 

## Сдача работы

Сохраните ноутбук вместе с результатами ячеек, закоммитьте его в свою ветку и откройте запросы на слияние в обоих форках, как описано в инструкции к лабораторной работе.